In [ ]:
!pip install -q openai

In [ ]:
from pathlib import Path
from pprint import pprint
from pydantic import BaseModel
from typing import List

def print_response(response):
    print(f"Reponse id: {response.id}")
    print(f"Status: {response.status}")
    print(f"Input tokens: {response.usage.input_tokens} ({response.usage.input_tokens_details.cached_tokens} cached) | Output tokens: {response.usage.output_tokens} ({response.usage.output_tokens_details.reasoning_tokens} reasoning)")
    pprint(response.output)

    print()
    print(f"{'=' * 20} [Text] {'=' * 20}")
    print(response.output_text)

In [ ]:
from google.colab import userdata
from openai import OpenAI

api_key = userdata.get('OPENROUTER_API_KEY')
client = OpenAI(api_key=api_key, base_url="https://openrouter.ai/api/v1")

## Contextual memory

The model can remember information within the same conversation (session), but once it ends, the memory is lost.

In [ ]:
generate_class_prompt = "Generate a simple Python class that represents a person."
generate_instances_prompt = "Generate three instances of the person class that represent a family - mother, father and their child."

In [ ]:
generate_class_response = client.responses.create(
    model="gpt-5-nano",
    input=generate_class_prompt
)

In [ ]:
print_response(generate_class_response)

In [ ]:
generate_instances_response = client.responses.create(
    model="gpt-5-nano",
    input=[
        { "role": "user", "content": generate_class_prompt },
        *generate_class_response.output,
        { "role": "user", "content": generate_instances_prompt }
    ]
)

In [ ]:
print_response(generate_instances_response)

## Persistent memory

The model can remember information across different sessions, if we save it in a file and provide it as context to the model in the next session.

In [ ]:
class MemorableResponse(BaseModel):
    response: str
    facts_to_remember: List[str]

PATH_TO_FACTS = Path("/content/facts.txt")
def save_facts(response: MemorableResponse) -> None:
    with PATH_TO_FACTS.open(mode="a") as file:
        file.writelines(f"{fact}\n" for fact in response.facts_to_remember)

def read_facts() -> List[str]:
    with PATH_TO_FACTS.open(mode="r") as file:
        return file.readlines()

In [ ]:
# This is a typical example for CAG (Cache Augmented Generation)
# We take all the known external information and provide it to the AI model in advance

def ask(question: str, history = []):
    known_facts = read_facts()
    answer_response = client.responses.parse(
        model="gpt-5-nano",
        input=[
            { "role": "developer", "content": "You are a helpful AI assistant that specializes in day-to-day interactions. Keep track of various interesting facts about the user that you may use in future to provide better responses and be even more helpful! If a fact is already known, there is no need to include it in the output." },
            *history,
            { "role": "assistant", "content": f"Facts I know about the user: {';'.join(known_facts)}"},
            { "role": "user", "content": question }
        ],
        text_format=MemorableResponse
    )

    save_facts(answer_response.output_parsed)

    return answer_response

In [ ]:
greeting_prompt = "Hello, AI. My name is Tony. How do you feel today?"
ask_about_age_prompt = "I feel fine. The weather is sunny. How old are you? I am 24-years-old."
where_are_you_from_prompt = "And what about your location? I live in Bulgaria."

In [ ]:
greeting_response = ask(greeting_prompt)

In [ ]:
print_response(greeting_response)

In [ ]:
ask_about_age_response = ask(
    ask_about_age_prompt,
    history=[
        { "role": "user", "content": greeting_prompt },
        *greeting_response.output
    ]
)

In [ ]:
print_response(ask_about_age_response)

In [ ]:
where_are_you_from_response = ask(
    where_are_you_from_prompt,
    history=[
        { "role": "user", "content": greeting_prompt },
        *greeting_response.output,
        { "role": "user", "content": ask_about_age_prompt },
        *ask_about_age_response.output
    ]
)

In [ ]:
print_response(where_are_you_from_response)

In [ ]:
# On the next day
# We start a new session, but the AI still remembers facts about the user, because we have saved them in a file and provided them as context to the model.

on_the_next_day_response = ask("Hello, AI! What are you doing today?")

In [ ]:
print_response(on_the_next_day_response)

## Semantic memory

The built-in knowledge of the model, which is based on the dataset it was trained on.

In [ ]:
tcp_ip_response = client.responses.create(
    model="gpt-5-nano",
    input="Explain how TCP/IP works. Use no more than 5 sentences"
)

In [ ]:
print_response(tcp_ip_response)

In [ ]:
world_history_response = client.responses.create(
    model="gpt-5-nano",
    input="In no more than 10 sentences, describe the history of the world up until now."
)

In [ ]:
print_response(world_history_response)

In [ ]:
# AI models are trained on a fixed dataset that includes information up until a certain point in time (the so-called "training cutoff date").
# Their semantic knowledge is limited to that dataset, and OOTB (without any additional tools) they do not have access to real-time information or updates about the world after that date.
# This explains why they cannot provide accurate information about the recent news.

actual_news_response = client.responses.create(
    model="gpt-5-nano",
    input="List the most recent and important news from the past week."
)

In [ ]:
print_response(actual_news_response)

## Working memory

Handles multiple reasoning steps and complex tasks that require the model to keep track of intermediate information and results.

In [ ]:
quadratic_equation_solving_response = client.responses.create(
    model="gpt-5.2",
    input="How to solve the quadratic equation 2*x^2-3*x+5=-1. What is the general solution plan?",
    reasoning={ "effort": "high" }
)

In [ ]:
print_response(quadratic_equation_solving_response)